In [1]:
import os
import torch
import torch.nn as nn
from collections import OrderedDict
from src.utilities.load_dataset import load_dataset
from src.utilities.calculate_accuracy import calculate_model_accuracy

In [5]:
os.makedirs("./_models", exist_ok=True)

modelpaths = [
    ("resnet50", "./_models/resnet50.onnx"),
    ("alexnet", "./_models/alexnet.onnx"),
    ("densenet121", "./_models/densenet121.onnx"),
    ("mobilenet_v3_small", "./_models/mobilenet_v3_small.onnx"),
    ("vgg16", "./_models/vgg16.onnx"),
]
for modelpath in modelpaths:
    if not os.path.exists(modelpath[1]):
        try:
            model = torch.hub.load("pytorch/vision:v0.13.1", modelpath[0], weights="IMAGENET1K_V2")
        except (ValueError, KeyError):
            model = torch.hub.load("pytorch/vision:v0.13.1", modelpath[0], weights="IMAGENET1K_V1")
        model.eval()
        torch.onnx.export(model, torch.ones(1,3,224,224), modelpath[1], verbose=True)

Using cache found in /home/lia-ws-094/.cache/torch/hub/pytorch_vision_v0.13.1
Using cache found in /home/lia-ws-094/.cache/torch/hub/pytorch_vision_v0.13.1
Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /home/lia-ws-094/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth
100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 233M/233M [00:29<00:00, 8.34MB/s]
Using cache found in /home/lia-ws-094/.cache/torch/hub/pytorch_vision_v0.13.1
Using cache found in /home/lia-ws-094/.cache/torch/hub/pytorch_vision_v0.13.1
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /home/lia-ws-094/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 30.8M/30.8M [00:03<00:00, 8.29MB/s]
Using cache found in /home/lia-ws-09

In [8]:
batch_size = 256
validation_dataset = load_dataset("test")

for modelpath in modelpaths:
    try:
        model = torch.hub.load("pytorch/vision:v0.13.1", modelpath[0], weights="IMAGENET1K_V2")
    except (ValueError, KeyError):
        model = torch.hub.load("pytorch/vision:v0.13.1", modelpath[0], weights="IMAGENET1K_V1")
    for params in model.parameters():
        params.requires_grad = False
    model.fc = nn.Sequential(OrderedDict([('fc', nn.Linear(model.fc.in_features, 3))]))
    model.eval()
    
    acc = calculate_model_accuracy(model, validation_dataset, batch_size)
    print(acc, modelpath[0])

Using cache found in /home/lia-ws-094/.cache/torch/hub/pytorch_vision_v0.13.1
  0%|                                                                                                                                                   | 0/11 [00:00<?, ?it/s]/home/lia-ws-094/shafigh/stitchnet/.stitchnet/lib/python3.11/site-packages/torch/nn/modules/conv.py:456: UserWarning: Applied workaround for CuDNN issue, install nvrtc.so (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:80.)
  return F.conv2d(input, weight, bias, self.stride,
  0%|                                                                                                                                                   | 0/11 [00:01<?, ?it/s]

torch.Size([256, 3, 224, 224]) tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]) [2 2 2 0 2 2 2 2 2 2 1 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 0 2 2 2 2 1 2 2 1 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 0 0 2 0 0 2 


Using cache found in /home/lia-ws-094/.cache/torch/hub/pytorch_vision_v0.13.1
Using cache found in /home/lia-ws-094/.cache/torch/hub/pytorch_vision_v0.13.1


AttributeError: 'AlexNet' object has no attribute 'fc'